# z023 - Préstamos personales en julio y control de calidad

**Parte A - Préstamos:** ¿la tasa de baja de los clientes con préstamos grandes sigue alta en 202107?
- `mprestamos_personales` en 3 grupos **calculados dentro de cada mes**: sin préstamo (= 0), con préstamo hasta el
  percentil 90 del mes, top 10% del mes. El percentil 90 se toma sobre **todos** los clientes del mes (~$100.000,
  como en 7_drifting); como ~78% tiene 0, el top 10% es casi la mitad de los que tienen préstamo.
- `cprestamos_personales`: 0, 1-2, 3 o más.
- En 202107 solo existe BAJA+1 (BAJA+2 necesitaría saber qué pasa en septiembre).

**Parte B - Calidad:** % de ceros y de NaN por variable y mes; se marcan los meses raros.

No modifica ningún otro archivo; guarda `z023_calidad.csv`.

In [1]:
# --- 1. Datos ---
import pandas as pd
import numpy as np

df = pd.read_parquet("../competencia_01.parquet")
MESES = [202103, 202104, 202105, 202106, 202107]
print(df.shape)

(983061, 155)


In [2]:
# --- 2. Grupos de préstamos (corte p90 recalculado en cada mes) ---
p90 = df.groupby("foto_mes")["mprestamos_personales"].quantile(0.9)   # un corte por mes
print("Percentil 90 por mes:"); print(p90.round(0))

corte = df["foto_mes"].map(p90)                    # a cada fila le pongo el corte de su mes
df["grupo_monto"] = np.select(
    [df["mprestamos_personales"] == 0, df["mprestamos_personales"] <= corte],
    ["1 sin préstamo", "2 hasta p90"], default="3 top 10%")

df["grupo_cant"] = pd.cut(df["cprestamos_personales"], bins=[-1, 0, 2, np.inf],
                          labels=["0", "1-2", "3 o más"])

Percentil 90 por mes:
foto_mes
202103     94750.0
202104     99278.0
202105    100412.0
202106     99894.0
202107     96357.0
202108     95658.0
Name: mprestamos_personales, dtype: float64


In [3]:
# --- 3. Tabla: clientes, BAJA+1, BAJA+2 y tasas por grupo y mes ---
def tabla(grupo):
    d = df[df["foto_mes"].isin(MESES)]
    t = d.groupby(["foto_mes", grupo], observed=True).agg(
        clientes=("numero_de_cliente", "size"),
        baja1=("clase_ternaria", lambda s: (s == "BAJA+1").sum()),
        baja2=("clase_ternaria", lambda s: (s == "BAJA+2").sum()))
    t["tasa_baja1_%"] = (t["baja1"] / t["clientes"] * 100).round(2)
    t["tasa_baja2_%"] = (t["baja2"] / t["clientes"] * 100).round(2)
    # en 202107 BAJA+2 no existe: lo dejo vacío en vez de 0 para no confundir
    jul = t.index.get_level_values("foto_mes") == 202107
    t.loc[jul, ["baja2", "tasa_baja2_%"]] = np.nan
    return t

t_monto = tabla("grupo_monto")
t_monto

clientes  baja1   baja2  tasa_baja1_%  tasa_baja2_%
foto_mes grupo_monto                                                        
202103   1 sin préstamo    127046    962   871.0          0.76          0.69
         2 hasta p90        19564     46    72.0          0.24          0.37
         3 top 10%          16290     11    17.0          0.07          0.10
202104   1 sin préstamo    127188    902  1053.0          0.71          0.83
         2 hasta p90        19767     46    66.0          0.23          0.33
         3 top 10%          16329     16    20.0          0.10          0.12
202105   1 sin préstamo    127863   1083   769.0          0.85          0.60
         2 hasta p90        19528     44    72.0          0.23          0.37
         3 top 10%          16377     16    29.0          0.10          0.18
202106   1 sin préstamo    128527    798   856.0          0.62          0.67
         2 hasta p90        19175     50   119.0          0.26          0.62
         3 top 10%          16412     26   123.0          0.16          0.75
202107   1 sin préstamo    129320    917     NaN          0.71           NaN
         2 hasta p90        18593    118     NaN          0.63           NaN
         3 top 10%          16435     68     NaN          0.41           NaN

In [4]:
t_cant = tabla("grupo_cant")
t_cant

clientes  baja1   baja2  tasa_baja1_%  tasa_baja2_%
foto_mes grupo_cant                                                     
202103   0             127046    962   871.0          0.76          0.69
         1-2            23823     45    57.0          0.19          0.24
         3 o más        12031     12    32.0          0.10          0.27
202104   0             127188    902  1053.0          0.71          0.83
         1-2            24163     38    61.0          0.16          0.25
         3 o más        11933     24    25.0          0.20          0.21
202105   0             127863   1083   769.0          0.85          0.60
         1-2            24355     43    66.0          0.18          0.27
         3 o más        11550     17    35.0          0.15          0.30
202106   0             128527    798   856.0          0.62          0.67
         1-2            24513     44    65.0          0.18          0.27
         3 o más        11074     32   177.0          0.29          1.60
202107   0             129320    917     NaN          0.71           NaN
         1-2            24518     42     NaN          0.17           NaN
         3 o más        10510    144     NaN          1.37           NaN

In [5]:
# --- 4. Resumen compacto: tasa BAJA+1 por mes (filas = grupo) ---
print("Tasa BAJA+1 (%) por monto:")
print(t_monto["tasa_baja1_%"].unstack("foto_mes"))
print(); print("Tasa BAJA+1 (%) por cantidad:")
print(t_cant["tasa_baja1_%"].unstack("foto_mes"))
print(); print("Tasa BAJA+2 (%) por monto:")
print(t_monto["tasa_baja2_%"].unstack("foto_mes").drop(columns=202107))

Tasa BAJA+1 (%) por monto:
foto_mes        202103  202104  202105  202106  202107
grupo_monto                                           
1 sin préstamo    0.76    0.71    0.85    0.62    0.71
2 hasta p90       0.24    0.23    0.23    0.26    0.63
3 top 10%         0.07    0.10    0.10    0.16    0.41

Tasa BAJA+1 (%) por cantidad:
foto_mes    202103  202104  202105  202106  202107
grupo_cant                                        
0             0.76    0.71    0.85    0.62    0.71
1-2           0.19    0.16    0.18    0.18    0.17
3 o más       0.10    0.20    0.15    0.29    1.37

Tasa BAJA+2 (%) por monto:
foto_mes        202103  202104  202105  202106
grupo_monto                                   
1 sin préstamo    0.69    0.83    0.60    0.67
2 hasta p90       0.37    0.33    0.37    0.62
3 top 10%         0.10    0.12    0.18    0.75


In [6]:
# --- 5. Parte B: % de ceros y NaN por variable y mes ---
excluir = ["numero_de_cliente", "foto_mes", "clase_ternaria", "grupo_monto", "grupo_cant"]
vars_ = [c for c in df.columns if c not in excluir]
g = df.groupby("foto_mes")[vars_]
pct_cero = (g.agg(lambda s: (s == 0).mean()) * 100).round(2)     # filas = mes, columnas = variable
pct_nan  = (g.agg(lambda s: s.isna().mean()) * 100).round(2)

calidad = pd.concat({"pct_cero": pct_cero.T, "pct_nan": pct_nan.T}, axis=1)   # filas = variable
calidad.to_csv("z023_calidad.csv", sep=";", decimal=",", encoding="utf-8-sig", index_label="variable")
calidad.head()

pct_cero                                    pct_nan         \
foto_mes             202103 202104 202105 202106 202107 202108  202103 202104   
active_quarter         1.43   1.31   1.27   1.23   1.25   1.34     0.0    0.0   
cliente_vip           99.74  99.74  99.73  99.44  99.73  99.72     0.0    0.0   
internet              95.96  95.43  95.57  96.31  97.41  96.82     0.0    0.0   
cliente_edad           0.00   0.00   0.00   0.00   0.00   0.00     0.0    0.0   
cliente_antiguedad     0.00   0.00   0.00   0.00   0.00   0.00     0.0    0.0   

                                                
foto_mes           202105 202106 202107 202108  
active_quarter        0.0    0.0    0.0    0.0  
cliente_vip           0.0    0.0    0.0    0.0  
internet              0.0    0.0    0.0    0.0  
cliente_edad          0.0    0.0    0.0    0.0  
cliente_antiguedad    0.0    0.0    0.0    0.0

In [7]:
# --- 6. Meses raros ---
# a) toda en cero o toda en NaN un mes, pero no en todos los meses
# b) salto de más de 30 puntos en % de ceros o % de NaN respecto del mes anterior
alertas = []
for nombre, t in [("cero", pct_cero), ("nan", pct_nan)]:
    for v in vars_:
        s = t[v]
        for m in s.index[(s >= 99.9) & (s.min() < 99.9)]:
            alertas.append({"variable": v, "tipo": nombre, "foto_mes": m, "motivo": f"{s[m]}% (otros meses mín {s.min()}%)"})
        salto = s.diff().abs()
        for m in salto.index[salto > 30]:
            alertas.append({"variable": v, "tipo": nombre, "foto_mes": m,
                            "motivo": f"salto {s.shift()[m]}% -> {s[m]}%"})
alertas = pd.DataFrame(alertas).drop_duplicates(["variable", "tipo", "foto_mes"])
print(len(alertas), "alertas")
alertas

18 alertas


,variable,tipo,foto_mes,motivo
0,mpayroll2,cero,202103,99.9% (otros meses mín 99.88%)
1,mpayroll2,cero,202104,99.91% (otros meses mín 99.88%)
2,mpayroll2,cero,202105,99.91% (otros meses mín 99.88%)
3,mpayroll2,cero,202107,99.95% (otros meses mín 99.88%)
4,mpayroll2,cero,202108,99.94% (otros meses mín 99.88%)
5,cpayroll2_trx,cero,202103,99.9% (otros meses mín 99.88%)
6,cpayroll2_trx,cero,202104,99.91% (otros meses mín 99.88%)
7,cpayroll2_trx,cero,202105,99.91% (otros meses mín 99.88%)
8,cpayroll2_trx,cero,202107,99.95% (otros meses mín 99.88%)
9,cpayroll2_trx,cero,202108,99.94% (otros meses mín 99.88%)
